# Sentiment Analysis Gradio App (Kaggle)

An interactive **Gradio web app** that loads the trained IMDB models
(`ag_tabular_imdb/`) and predicts whether textual comments are
**positive** or **negative**.

Features:
- Single-comment analysis with a positive/negative verdict + confidence gauge
- Model selector (Tabular ensemble, or an averaging Ensemble if both are loaded)
- Batch analysis: paste many comments or upload a CSV, get a results table + download
- Runs inside Kaggle via Gradio's public share link

> **Kaggle setup**
> - Add the saved model folders (`ag_tabular_imdb/`) as a dataset input,
>   *or* run in the same session right after training so the folders exist in the working dir.
> - Turn **Internet ON** (Settings) so Gradio can create the public `share=True` link.


## 0. Install dependencies

In [1]:
import importlib.util
if importlib.util.find_spec("autogluon") is None:
    !pip install -q -U pip
    !pip install -q autogluon.tabular autogluon.multimodal
if importlib.util.find_spec("gradio") is None:
    !pip install -q gradio
print("Dependencies ready.")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 6.5 MB/s eta 0:00:0000:0100:01
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
tpot 1.1.0 requires dill>=0.3.9, but you have dill 0.3.8 which is incompatible.
torchaudio 2.10.0+cpu requ

## 1. Imports & environment

In [2]:
import os
import re
import glob
import numpy as np
import pandas as pd
import gradio as gr

try:
    import torch
    GPU_AVAILABLE = torch.cuda.is_available()
except Exception:
    GPU_AVAILABLE = False

print("Gradio version:", gr.__version__)
print("GPU available:", GPU_AVAILABLE)


Gradio version: 5.50.0
GPU available: False


## 2. Locate & load the trained models

In [3]:
def find_path(patterns):
    for pat in patterns:
        hits = glob.glob(pat, recursive=True)
        if hits:
            return hits[0]
    return None
    
TABULAR_DIR = find_path(["/kaggle/input/**/imdb-sentiment-classification/ag_tabular_imdb", "ag_tabular_imdb"])

print("Tabular model dir   :", TABULAR_DIR)

predictors = {}

if TABULAR_DIR is not None:
    from autogluon.tabular import TabularPredictor
    predictors["Tabular"] = TabularPredictor.load(TABULAR_DIR)
    print("Loaded TabularPredictor.")

if not predictors:
    raise FileNotFoundError(
        "No models found. Add ag_tabular_imdb/ as a Kaggle "
        "input, or run this in the same session as training."
    )

# Build the list of selectable options for the UI
MODEL_CHOICES = list(predictors.keys())
if len(predictors) > 1:
    MODEL_CHOICES.append("Ensemble (average)")
print("Model choices for the app:", MODEL_CHOICES)


Tabular model dir   : /kaggle/input/notebooks/mishasondhi/imdb-sentiment-classification/ag_tabular_imdb
Loaded TabularPredictor.
Model choices for the app: ['Tabular']


## 3. Prediction logic

In [4]:
POS = "positive"

def clean_text(text: str) -> str:
    text = re.sub(r"<br\s*/?>", " ", str(text))
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def _proba_positive(predictor, frame: pd.DataFrame) -> np.ndarray:
    """Return P(positive) as a 1-D array for a dataframe with a 'review' column."""
    proba_df = predictor.predict_proba(frame)
    if POS in proba_df.columns:
        return proba_df[POS].values
    return proba_df.iloc[:, -1].values

def predict_proba_positive(texts, model_choice: str) -> np.ndarray:
    """Core inference: list[str] -> np.array of P(positive)."""
    frame = pd.DataFrame({"review": [clean_text(t) for t in texts]})
    if model_choice == "Ensemble (average)":
        stacks = [_proba_positive(p, frame) for p in predictors.values()]
        return np.mean(np.column_stack(stacks), axis=1)
    return _proba_positive(predictors[model_choice], frame)

def label_from_proba(p: float) -> str:
    return "positive" if p >= 0.5 else "negative"


## 4. UI callback functions

In [5]:
def analyze_single(comment: str, model_choice: str):
    comment = (comment or "").strip()
    if not comment:
        return "Please enter a comment.", {}

    p_pos = float(predict_proba_positive([comment], model_choice)[0])
    label = label_from_proba(p_pos)
    confidence = p_pos if label == "positive" else 1 - p_pos

    emoji = "\U0001F600" if label == "positive" else "\U0001F614"  # smiley / sad
    verdict = f"{emoji}  **{label.upper()}**  —  {confidence*100:.1f}% confident"

    # gr.Label expects {class: prob}
    label_scores = {"positive": p_pos, "negative": 1 - p_pos}
    return verdict, label_scores


def analyze_batch_text(block: str, model_choice: str):
    lines = [ln.strip() for ln in (block or "").splitlines() if ln.strip()]
    if not lines:
        return pd.DataFrame(columns=["comment", "prediction", "P(positive)"]), None

    probs = predict_proba_positive(lines, model_choice)
    out = pd.DataFrame({
        "comment": lines,
        "prediction": [label_from_proba(p) for p in probs],
        "P(positive)": np.round(probs, 4),
    })
    csv_path = "batch_predictions.csv"
    out.to_csv(csv_path, index=False)
    return out, csv_path


def analyze_csv(file, text_column, model_choice: str):
    if file is None:
        return pd.DataFrame(), None, "Upload a CSV first."
    data = pd.read_csv(file.name)
    col = (text_column or "").strip()
    if col not in data.columns:
        # fall back to the first text-like column
        col = data.columns[0]
    texts = data[col].astype(str).tolist()

    probs = predict_proba_positive(texts, model_choice)
    data = data.copy()
    data["prediction"] = [label_from_proba(p) for p in probs]
    data["P(positive)"] = np.round(probs, 4)

    csv_path = "uploaded_predictions.csv"
    data.to_csv(csv_path, index=False)
    summary = (f"Scored {len(data)} rows using column '{col}'. "
               f"Positive: {(data.prediction=='positive').sum()} | "
               f"Negative: {(data.prediction=='negative').sum()}")
    return data.head(200), csv_path, summary


## 5. Build the Gradio interface

In [6]:
EXAMPLES = [
    "This movie was an absolute masterpiece, the acting was superb!",
    "Complete waste of time. Terrible plot and even worse acting.",
    "I expected to hate it but it completely won me over.",
    "Great cast, gorgeous cinematography... and a plot that goes nowhere.",
]

default_model = MODEL_CHOICES[-1] if "Ensemble (average)" in MODEL_CHOICES else MODEL_CHOICES[0]

with gr.Blocks(title="Sentiment Analyzer", theme=gr.themes.Soft()) as demo:
    gr.Markdown(
        "# \U0001F3AC Comment Sentiment Analyzer\n"
        "Predict whether textual comments are **positive** or **negative** "
        "using models trained on the IMDB reviews dataset."
    )

    model_selector = gr.Radio(
        choices=MODEL_CHOICES, value=default_model, label="Model"
    )

    with gr.Tab("Single comment"):
        comment_in = gr.Textbox(
            lines=4, label="Comment", placeholder="Type or paste a comment here..."
        )
        analyze_btn = gr.Button("Analyze", variant="primary")
        verdict_out = gr.Markdown()
        score_out = gr.Label(num_top_classes=2, label="Class probabilities")
        gr.Examples(examples=EXAMPLES, inputs=comment_in)

        analyze_btn.click(
            analyze_single,
            inputs=[comment_in, model_selector],
            outputs=[verdict_out, score_out],
        )

    with gr.Tab("Batch (paste lines)"):
        gr.Markdown("Paste **one comment per line**, then analyze all at once.")
        batch_in = gr.Textbox(lines=10, label="Comments (one per line)")
        batch_btn = gr.Button("Analyze batch", variant="primary")
        batch_table = gr.Dataframe(label="Results")
        batch_file = gr.File(label="Download results (CSV)")
        batch_btn.click(
            analyze_batch_text,
            inputs=[batch_in, model_selector],
            outputs=[batch_table, batch_file],
        )

    with gr.Tab("Upload CSV"):
        gr.Markdown("Upload a CSV and name the text column to score every row.")
        csv_in = gr.File(label="CSV file", file_types=[".csv"])
        col_in = gr.Textbox(value="review", label="Text column name")
        csv_btn = gr.Button("Analyze CSV", variant="primary")
        csv_summary = gr.Markdown()
        csv_table = gr.Dataframe(label="Results (first 200 rows)")
        csv_file = gr.File(label="Download full results (CSV)")
        csv_btn.click(
            analyze_csv,
            inputs=[csv_in, col_in, model_selector],
            outputs=[csv_table, csv_file, csv_summary],
        )

    gr.Markdown(
        "_Models: AutoGluon Tabular ensemble"
        + (" + fine-tuned Transformer" if "Transformer" in predictors else "")
        + ". Built on the IMDB sentiment dataset._"
    )

print("Interface built.")


/tmp/ipykernel_58/1153406118.py:10: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(title="Sentiment Analyzer", theme=gr.themes.Soft()) as demo:


Interface built.


## 6. Launch the app

`share=True` gives a public URL that works from inside Kaggle. The link stays live while this
cell is running — stop the cell (or the session) to shut the app down.

> If the share link doesn't appear, confirm **Internet is ON** in Kaggle Settings.

In [7]:
demo.launch(share=True)


* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://e56e5bc42bf08351b7.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## Notes

- **Which model to pick?** The Ensemble (if available) is usually most accurate; the Transformer
  alone is strongest on nuanced/sarcastic text; the Tabular ensemble is fastest.
- **CSV upload**: set the *Text column name* to match your file (defaults to `review`).
- **Deploy elsewhere**: the same `demo` object can be pushed to Hugging Face Spaces — just add a
  `requirements.txt` with `gradio`, `autogluon.tabular`, `autogluon.multimodal`.
- To free memory, load only one model (comment out the other in Section 2).
